In [ ]:
# ===== Cell 1: Setup & Imports =====

import sys
import os
from pathlib import Path

# Ensure the project root is in the Python path
PROJECT_ROOT = Path('../')
sys.path.insert(0, str(PROJECT_ROOT))

from src.news_processor import NewsProcessor
from src.data_processor import FeatureProcessor

DATA_DIR = PROJECT_ROOT / 'data'
RAW_NEWS_DIR = DATA_DIR / 'raw_data' / 'news_data'
OUTPUT_JSONL = DATA_DIR / 'processed_data' / 'daily_news_features.jsonl'

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'News data dir: {RAW_NEWS_DIR}')
print(f'Output JSONL: {OUTPUT_JSONL}')

## Step 1: Define Stock Universe

In [ ]:
# ===== Cell 2: Define stock list =====
# These should match the stocks used in your technical dataset

# Auto-detect from available news files
available_news = sorted([f.stem.replace('_NEWS', '') for f in RAW_NEWS_DIR.glob('*_NEWS.jsonl')])
print(f'Found {len(available_news)} tickers with news data:')
print(available_news)

# Use all available tickers (or filter as needed)
STOCKS = available_news

## Step 2: Process News into JSONL Features

In [ ]:
# ===== Cell 3: Process all tickers =====
# This will:
#   1. Load each ticker's raw news JSONL
#   2. For each day: filter duplicates, rank by relevance, pick top article
#   3. Extract 768-dim FinBERT embedding + 4 sentiment scores
#   4. Save to a single JSONL file for fast lookup
#
# WARNING: This requires GPU and takes significant time for the first run.

processor = NewsProcessor()  # auto-detects GPU

processor.process_all_tickers(
    stocks=STOCKS,
    news_dir=RAW_NEWS_DIR,
    output_jsonl_path=OUTPUT_JSONL
)

print(f'\nNews features saved to: {OUTPUT_JSONL}')

## Step 3: Generate Dual-Path Dataset

In [ ]:
# ===== Cell 4: Build the dual-path dataset =====
# This creates a .pt file with separate tensors:
#   x_tech: [M, 17]  (13 technical + 4 sentiment through VSN)
#   x_emb:  [M, 768] (news embeddings through EmbeddingBranch)
#   static_id: stock index
#   y: label (0=Sell, 1=Neutral, 2=Buy)

feature_proc = FeatureProcessor(base_dir=str(DATA_DIR))

# Dataset parameters (match your existing tech+ER dataset settings)
DATES = {
    'start': '2021-01-01',
    'end': '2024-12-31'
}
M = 60           # Lookback window (trading days)
T = 5            # Prediction horizon (trading days)
MULTIPLIER = 1.5 # ATR multiplier for labeling

feature_proc.generate_dual_path_dataset(
    stocks=STOCKS,
    dates=DATES,
    M=M,
    T=T,
    multiplier=MULTIPLIER,
    news_jsonl_path=str(OUTPUT_JSONL)
)

## Step 4: Verify Dataset

In [ ]:
# ===== Cell 5: Verify the generated dataset =====
import torch
import numpy as np

ds_path = DATA_DIR / 'processed_data' / 'dual_path_dataset.pt'
samples = torch.load(ds_path, weights_only=False)

print(f'Total samples: {len(samples)}')
print(f'Sample keys: {list(samples[0].keys())}')
print()

s = samples[0]
print(f'x_tech shape: {s["x_tech"].shape}  (expected: [{M}, 17])')
print(f'x_emb shape:  {s["x_emb"].shape}   (expected: [{M}, 768])')
print(f'static_id:    {s["static_id"]}')
print(f'y (label):    {s["y"]}')
print(f'metadata:     {s["metadata"]}')

# Check embedding coverage
has_news = sum(1 for s in samples if np.any(s['x_emb'] != 0))
print(f'\nSamples with at least one non-zero embedding day: {has_news}/{len(samples)} ({has_news/len(samples):.1%})')

# Label distribution
labels = [s['y'] for s in samples]
counts = np.bincount(labels, minlength=3)
print(f'Label distribution: Sell={counts[0]}, Neutral={counts[1]}, Buy={counts[2]}')